# Validation Test (comparison between XDG and WNLT) - Oscillating droplet

This notebook and the corresponding simulation setup notebook (OscillatingDroplet_DACH_Comparison_Run.ipynb) are part of published results (cf. V.A.) found in *From weakly to strongly nonlinear viscous drop shape oscillations: An analytical and numerical study* (https://doi.org/10.1103/PhysRevFluids.9.063601). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Initialization tasks

Loading the `XNSE_Solver` and additional namespace:

In [ ]:
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using NUnit.Framework;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;

Overview on the available *Execution Queues* (aka. *Batch Processors*, aka. *Batch System*); these e.g. Linux HPC clusters on which compute jobs can be executed.

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [4]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

For this notebook (which is part of the BoSSS validation tests), a *default queue* is selected to run all jobs in the convergence study:

In [5]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


Initialization of the Workflow management; there `OscillatingDropletPaper` is the project name which is used name all computations (aka. sessions):

In [6]:
wmg.Init("OscillatingDropletPaper", myBatch);

Project name is set to 'OscillatingDropletPaper'.
Creating database '\\dc3\userspace\smuda\hpccluster\OscillatingDropletPaper'.


In [7]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [ ]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up tp 7.0), otherwise they will run only up to 0.5

## Sessions

In [9]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

## Verification of Initial Value data

Initial values and parameters for the simulation (intial droplet shape, Ohnesorg number, initial velocity)
were specified by project partner (TU Graz, Group Prof. Brenn).
Details can be found in the corresponding publication.

First, it is verified that the initial values chosen here actually match the specification.

In [61]:
//MultidimensionalArray[] ReferenceData = new MultidimensionalArray[5];
int[] modes = new int[] { 2, 3, 4 };
List<string> cases = new List<string>();
int nCase = 0;
// add simulations for mode 2
if(modes.Contains(2)) {
    cases.Add("m2_Oh01_eta04");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m2_Oh01_eta02");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m2_Oh01_eta01");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
// add simulations for mode 3
if(modes.Contains(3)) {
    cases.Add("m3_Oh01_eta04");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    // cases.Add("m3_Oh01_eta03");
    // Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    // nCase++;
    cases.Add("m3_Oh01_eta015");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
// add simulations for mode 4
if(modes.Contains(4)) {
    cases.Add("m4_Oh01_eta04");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh01_eta01");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
    cases.Add("m4_Oh056_eta005");
    Console.WriteLine($"case {nCase+1}: {cases[nCase]}");
    nCase++;
}
int numCases = cases.Count;
numCases

case 1: m2_Oh01_eta04
case 2: m2_Oh01_eta02
case 3: m2_Oh01_eta01
case 4: m3_Oh01_eta04
case 5: m3_Oh01_eta015
case 6: m4_Oh01_eta04
case 7: m4_Oh01_eta01
case 8: m4_Oh056_eta005


8

Load reference data for all cases; These files contain two columns, i.e. the azimuth angle and the respective droplet radius.

In [62]:
MultidimensionalArray[] ReferenceData = new MultidimensionalArray[numCases];
for(int iCase = 0; iCase < numCases; iCase++) {
    if(userName.Equals(@"FDY\jenkinsci"))  
        ReferenceData[iCase] = IMatrixExtensions.LoadFromTextFile($"surfaceDrop_{cases[iCase]}.txt");
    else 
        ReferenceData[iCase] = IMatrixExtensions.LoadFromTextFile($"../data/InitialValues/{cases[iCase].Substring(0,2)}/surfaceDrop_{cases[iCase]}.txt");
}

Analytical expressions for the reference data (see `setup.pdf`); this is to verify that the definition of Legendre
Functions resp. Polynomials in BoSSS actually matches the definition used by the TU Graz group.

In [32]:
Dictionary<string, Func<double, double>> casesRadius = new Dictionary<string, Func<double, double>>();

In [33]:
double caseR_m2eta04(double angle) {
   double radius = 0.966781 + 0.4*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta04", caseR_m2eta04);

In [34]:
double caseR_m2eta02(double angle) {
   double radius = 0.991848 + 0.2*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta02", caseR_m2eta02);

In [35]:
double caseR_m2eta01(double angle) {
   double radius = 0.997981 + 0.1*SphericalHarmonics.MyLegendre(2,0,Math.Cos(angle));
   return radius; 
}
casesRadius.Add("m2_Oh01_eta01", caseR_m2eta01);

In [36]:
double caseR_m3eta04(double angle) {
    double radius = 0.977143 - 0.00598442*Math.Cos(angle) + 0.4*SphericalHarmonics.MyLegendre(3,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m3_Oh01_eta04", caseR_m3eta04);

In [37]:
double caseR_m3eta03(double angle) {
    double radius = 0.987143 - 0.00252468*Math.Cos(angle) + 0.3*SphericalHarmonics.MyLegendre(3,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m3_Oh01_eta03", caseR_m3eta03);

In [38]:
double caseR_m3eta015(double angle) {
    double radius = 0.996786 - 0.000315584*Math.Cos(angle) + 0.15*SphericalHarmonics.MyLegendre(3,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m3_Oh01_eta015", caseR_m3eta015);

In [39]:
double caseR_m4eta04(double angle) {
    double radius = 0.981839 + 0.4*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta04", caseR_m4eta04);

In [40]:
double caseR_m4eta01(double angle) {
    double radius = 0.998883 + 0.1*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh01_eta01", caseR_m4eta01);

In [41]:
double caseR_m4eta005(double angle) {
    double radius = 0.999721 + 0.05*SphericalHarmonics.MyLegendre(4,0,Math.Cos(angle));
    return radius; 
}
casesRadius.Add("m4_Oh056_eta005", caseR_m4eta005);

Conversion to cartesian coordinates in order to match the data and verification against analytical expression:

In [63]:
double[][] refX = new double[numCases][];
double[][] refZ = new double[numCases][];
double[][] caseX = new double[numCases][];
double[][] caseZ = new double[numCases][];
for(int iCase = 0; iCase < numCases; iCase++) {
// int testcase = 4;
// for(int iCase = testcase; iCase < testcase+1; iCase++) {
    double[] angle = ReferenceData[iCase].GetColumn(0);
    double[] radius = ReferenceData[iCase].GetColumn(1);
       
    double RadiusErrorNorm = 0.0;
    int I = angle.Length;
    double[] x1 = new double[I], z1 = new double[I];
    double[] cx1 = new double[I], cz1 = new double[I];
    for(int i = 0; i < I; i++) {
        x1[i] = Math.Sin(angle[i])*radius[i];
        z1[i] = Math.Cos(angle[i])*radius[i];
        
        double radius_expr = casesRadius[cases[iCase]](angle[i]);
        RadiusErrorNorm += (radius[i] - radius_expr).Pow2();     
        
        cx1[i] = Math.Sin(angle[i])*radius_expr;
        cz1[i] = Math.Cos(angle[i])*radius_expr;
    } 
    refX[iCase] = x1;
    refZ[iCase] = z1;
    caseX[iCase] = cx1;
    caseZ[iCase] = cz1;

    RadiusErrorNorm = RadiusErrorNorm.Sqrt();
    Console.WriteLine($"Comparison error for radius in case {iCase + 1}: {RadiusErrorNorm}");
    // Note: since the factors in `setup.pdf` are only provided up to 6 digits, an error threshold of 1e-5 seems reasonable.
    //Assert.LessOrEqual(RadiusErrorNorm, 1e-5, "Error in comparing reference data against Legendre polynomials in BoSSS");
}

Comparison error for radius in case 1: 8.451542560852182E-07
Comparison error for radius in case 2: 6.761234036477466E-06
Comparison error for radius in case 3: 8.464947065400347E-07
Comparison error for radius in case 4: 2.540302895441871E-06
Comparison error for radius in case 5: 5.078951284690835E-06
Comparison error for radius in case 6: 6.9956707656638065E-06
Comparison error for radius in case 7: 1.8686025685177474E-06
Comparison error for radius in case 8: 8.39441090790297E-06


### Plot of Reference Data

In [43]:
int refCase = 0;
Plot(refX[refCase], refZ[refCase], "Ref-Case", ". black",
    caseX[refCase], caseZ[refCase], "Case", ". blue")

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -1.5 
 
 
 
 
 -1 
 
 
 
 
 -0.5 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 
 
 
 
 Ref-Case 
 
 
 Ref-Case 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M716.2,34.7 L758.4,34.7 M53.9,41.0 L66.2,41.0 L78.5,41.1 L90.8,41.2 L103.1,41.4 L115.4,41.6
 L127.7,41.8 L139.9,42.1 L152.1,42.5 L164.2,42.9 L176.4,43.3 L188.4,43.8 L200.5,44.4 L212.4,44.9
 L224.3,45.6 L236.2,46.2 L248.0,46.9 L259.7,47.7 L271.4,48.5 L282.9,49.3 L294.4,50.2 L305.8,51.1
 L317.1,52.1 L328.4,53.1 L339.5,54.2 L350.5,55.2 L361.4,56.4 L372.2,57.5 L383.0,58.7 L393.5,60.0
 L404.0,61.3 L414.4,62.6 L424.6,63.9 L434.7,65.3 L444.7,66.8 L454.5,68.2 L464.3,69.7 L473.8,71.2
 L483.3,72.8 L492.6,74.4 L501.7,76.0 L510.7,77.6 L519.6,79.3 L528.3,81.0 L536.9,82.7 L545.3,84.5
 L553.5,86.3 L561.6,88.1 L569.5,89.9 L577.3,91.7 L584.9,93.6 L592.4,95.5 L599.7,97.4 L606.8,99.3
 L613.8,101.3 L620.6,103.3 L627.3,105.3 L633.7,107.3 L640.0,109.3 L646.2,111.3 L652.2,113.4 L658.0,115.4
 L663.6,117.5 L669.1,119.6 L674.4,121.7 L679.6,123.8 L684.6,125.9 L689.4,128.0 L694.1,130.1 L698.6,132.2
 L703.0,134.4 L707.2,136.5 L711.2,138.7 L715.1,140.8 L718.8,143.0 L722.4,145.1 L725.8,147.3 L729.1,149.4
 L732.2,151.6 L735.2,153.7 L738.0,155.9 L740.7,158.0 L743.3,160.2 L745.7,162.3 L748.0,164.4 L750.1,166.6
 L752.2,168.7 L754.1,170.8 L755.8,172.9 L757.5,175.0 L759.0,177.1 L760.4,179.2 L761.7,181.3 L762.9,183.3
 L764.0,185.4 L764.9,187.4 L765.8,189.5 L766.6,191.5 L767.3,193.5 L767.8,195.5 L768.3,197.5 L768.7,199.5
 L769.1,201.4 L769.3,203.4 L769.5,205.3 L769.6,207.3 L769.6,209.2 L769.5,211.1 L769.4,213.0 L769.3,214.8
 L769.0,216.7 L768.7,218.5 L768.4,220.4 L768.0,222.2 L767.6,224.0 L767.1,225.7 L766.6,227.5 L766.1,229.3
 L765.5,231.0 L764.9,232.7 L764.3,234.5 L763.6,236.2 L762.9,237.8 L762.3,239.5 L761.5,241.2 L760.8,242.8
 L760.1,244.4 L759.4,246.1 L758.6,247.7 L757.9,249.3 L757.2,250.8 L756.4,252.4 L755.7,254.0 L755.0,255.5
 L754.3,257.0 L753.6,258.6 L752.9,260.1 L752.3,261.6 L751.6,263.1 L751.0,264.6 L750.4,266.0 L749.8,267.5
 L749.3,269.0 L748.8,270.4 L748.3,271.9 L747.8,273.3 L747.4,274.7 L747.0,276.2 L746.6,277.6 L746.3,279.0
 L746.0,280.4 L745.8,281.8 L745.6,283.2 L745.4,284.6 L745.2,286.0 L745.1,287.4 L745.1,288.8 L745.1,290.2
 L745.1,291.6 L745.1,293.0 L745.2,294.4 L745.4,295.8 L745.5,297.2 L745.7,298.6 L746.0,300.1 L746.3,301.5
 L746.6,302.9 L746.9,304.3 L747.3,305.7 L747.7,307.2 L748.2,308.6 L748.7,310.1 L749.2,311.5 L749.7,313.0
 L750.3,314.4 L750.9,315.9 L751.5,317.4 L752.1,318.9 L752.8,320.4 L753.5,321.9 L754.2,323.4 L754.9,324.9
 L755.6,326.5 L756.3,328.0 L757.0,329.6 L757.8,331.2 L758.5,332.8 L759.3,334.4 L760.0,336.0 L760.7,337.6
 L761.4,339.3 L762.1,340.9 L762.8,342.6 L763.5,344.3 L764.2,346.0 L764.8,347.7 L765.4,349.4 L766.0,351.1
 L766.5,352.9 L767.1,354.7 L767.5,356.5 L768.0,358.2 L768.4,360.1 L768.7,361.9 L769.0,363.7 L769.2,365.6
 L769.4,367.4 L769.5,369.3 L769.6,371.2 L769.6,373.1 L769.5,375.1 L769.3,377.0 L769.1,378.9 L768.8,380.9
 L768.4,382.9 L767.9,384.9 L767.4,386.9 L766.7,388.9 L765.9,390.9 L765.1,392.9 L764.1,395.0 L763.1,397.0
 L761.9,399.1 L760.6,401.2 L759.2,403.3 L757.7,405.4 L756.1,407.5 L754.3,409.6 L752.5,411.7 L750.5,413.8
 L748.3,415.9 L746.1,418.1 L743.7,420.2 L741.1,422.3 L738.5,424.5 L735.6,426.6 L732.7,428.8 L729.6,430.9
 L726.3,433.1 L722.9,435.2 L719.4,437.4 L715.7,439.5 L711.8,441.7 L707.8,443.8 L703.6,446.0 L699.3,448.1
 L694.8,450.2 L690.2,452.4 L685.4,454.5 L680.4,456.6 L675.3,458.7 L670.0,460.8 L664.5,462.9 L658.9,465.0
 L653.1,467.0 L647.2,469.1 L641.0,471.1 L634.7,473.1 L628.3,475.1 L621.7,477.1 L614.9,479.1 L608.0,481.0
 L600.8,483.0 L593.6,484.9 L58

### Matching of the Spherical Harmonics against the provided Data

In [44]:
Dictionary<string, Formula> casesPhi = new Dictionary<string, Formula>();

In [45]:
var Phi_m2eta04_Init = new Formula(
"Phi1",
false,
"using ilPSP.Utils; " + 
"double Phi1(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.966781*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.4*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"}");
casesPhi.Add("m2_Oh01_eta04", Phi_m2eta04_Init);

In [46]:
var Phi_m2eta02_Init = new Formula(
"Phi4",
false,
"using ilPSP.Utils; " + 
"double Phi4(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.991848*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.2*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m2_Oh01_eta02", Phi_m2eta02_Init);

In [47]:
var Phi_m2eta01_Init = new Formula(
"Phi4",
false,
"using ilPSP.Utils; " + 
"double Phi4(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.997981*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.1*SphericalHarmonics.MyRealSpherical(2, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m2_Oh01_eta01", Phi_m2eta01_Init);

In [48]:
var Phi_m3eta04_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    (0.977143 - 0.00598442*Math.Cos(theta))*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.4*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta04", Phi_m3eta04_Init);

In [49]:
var Phi_m3eta03_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    (0.987143 - 0.00252468*Math.Cos(theta))*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.3*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta03", Phi_m3eta03_Init);

In [50]:
var Phi_m3eta015_Init = new Formula(
"Phi2",
false,
"using ilPSP.Utils; " + 
"double Phi2(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    (0.996786 - 0.000315584*Math.Cos(theta))*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.15*SphericalHarmonics.MyRealSpherical(3, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m3_Oh01_eta015", Phi_m3eta015_Init);

In [51]:
var Phi_m4eta04_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.981839*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.4*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta04", Phi_m4eta04_Init);

In [52]:
var Phi_m4eta01_Init = new Formula(
"Phi3",
false,
"using ilPSP.Utils; " + 
"double Phi3(double[] X) { " +    
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.998883*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.1*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh01_eta01", Phi_m4eta01_Init);

In [53]:
var Phi_m4eta005_Init = new Formula(
"Phi5",
false,
"using ilPSP.Utils; " + 
"double Phi5(double[] X) { " + 
"    (double theta, double phi) = SphericalHarmonics.GetAngular(X); " + 
"    double R =    0.999721*SphericalHarmonics.MyRealSpherical(0, 0, theta, phi) " + 
"                +      0.05*SphericalHarmonics.MyRealSpherical(4, 0, theta, phi); " + 
"    return X.L2Norm() - R; " + 
"} ");
casesPhi.Add("m4_Oh056_eta005", Phi_m4eta005_Init);

In [64]:
for(int iCase = 0; iCase < numCases; iCase++) {
    double[] angle = ReferenceData[iCase].GetColumn(0);
    int I = angle.Length;
    
    double PhiErr = 0;
    for(int i = 0; i < I; i++) {
        double radius_expr = casesRadius[cases[iCase]](angle[i]);    
        double x1 = Math.Sin(angle[i])*radius_expr;
        double z1 = Math.Cos(angle[i])*radius_expr;
    
        PhiErr += casesPhi[cases[iCase]].Evaluate(new Vector(x1, 0, z1), 0.0).Abs();
    }
    Console.WriteLine($"Phi error for case {iCase}: {PhiErr}");
    Assert.LessOrEqual(PhiErr, 1e-10, "Level-Set function is not zero at desired surface.");
    
    Assert.IsTrue(casesPhi[cases[iCase]].Evaluate(new Vector(1e-5, 1e-5, 1e-5), 0.0) < 0, "Inside must be phase A/negative");
    Assert.IsTrue(casesPhi[cases[iCase]].Evaluate(new Vector(1e+1, 1e+1, 1e+1), 0.0) > 0, "Outside must be phase B/positive");
}

Phi error for case 0: 2.0650148258027912E-14
Phi error for case 1: 1.532107773982716E-14
Phi error for case 2: 1.0880185641326534E-14
Phi error for case 3: 2.1094237467877974E-14
Phi error for case 4: 1.2101430968414206E-14
Phi error for case 5: 3.530509218307998E-14
Phi error for case 6: 1.454392162258955E-14
Phi error for case 7: 1.2545520178264269E-14


### Initial Velocities

In [65]:
Dictionary<string, (IBoundaryAndInitialData velX, IBoundaryAndInitialData velY, IBoundaryAndInitialData velZ)> analyticVel = new Dictionary<string, (IBoundaryAndInitialData velX, IBoundaryAndInitialData velY, IBoundaryAndInitialData velZ)>();

for(int iCase = 0; iCase < numCases; iCase++) {
    MultidimensionalArray polVel;
    MultidimensionalArray radVel;
    if(userName.Equals(@"FDY\jenkinsci"))  {
        polVel = IMatrixExtensions.LoadFromTextFile($"polarVel_{cases[iCase]}.txt");
        radVel = IMatrixExtensions.LoadFromTextFile($"radialVel_{cases[iCase]}.txt");
    } else {
        polVel = IMatrixExtensions.LoadFromTextFile($"../data/InitialValues/{cases[iCase].Substring(0,2)}/polarVel_{cases[iCase]}.txt");
        radVel = IMatrixExtensions.LoadFromTextFile($"../data/InitialValues/{cases[iCase].Substring(0,2)}/radialVel_{cases[iCase]}.txt");
    }
    Assert.IsTrue(ilPSP.Utils.ArrayTools.ListEquals(polVel.GetColumn(0), radVel.GetColumn(0)));
    Assert.IsTrue(ilPSP.Utils.ArrayTools.ListEquals(polVel.GetColumn(1), radVel.GetColumn(1)));
    
    double[] radiusS = polVel.GetColumn(0);
    double[] anglesS = polVel.GetColumn(1);
    double[] polVelS = polVel.GetColumn(2);
    double[] radVelS = radVel.GetColumn(2);
    
    var velX = new BoSSS.Application.XNSE_Solver.SpecificSolutions.PolarAxiallySymmetricInitialValues() { VelocityComponent = 0 };
    velX.SetData(anglesS, radiusS, polVelS, radVelS);
    var velY = new BoSSS.Application.XNSE_Solver.SpecificSolutions.PolarAxiallySymmetricInitialValues() { VelocityComponent = 1 };
    velY.SetData(anglesS, radiusS, polVelS, radVelS);
    var velZ = new BoSSS.Application.XNSE_Solver.SpecificSolutions.PolarAxiallySymmetricInitialValues() { VelocityComponent = 2 };
    velZ.SetData(anglesS, radiusS, polVelS, radVelS);
    
    analyticVel.Add(cases[iCase], (velX, velY, velZ));
}

## Grid Creation

### Quater-Domain grid
(Symmetry planes at $x = 0$ and $y = 0$)

In [12]:
int[] Resolutions = new int[] { 7 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];
double scale = 1.0;
for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"OscillatingDroplet3D_{Res}x{Res}x{2*Res}_wallBC_quarterDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    //cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(0, 3*scale, Res + 1);
        double[] yNodes = xNodes;
        double[] zNodes = GenericBlas.Linspace(-3*scale, 3*scale, Res*2 + 1);
        
        var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if(X.x.Abs() <= 1e-8 || X.y.Abs() <= 1.0e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            else
                ret = IncompressibleBcType.Wall.ToString();
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Creating database 'C:\BoSSS-localJobs\OscillatingDropletPaper'.
Grid Edge Tags changed.


## Setup of control objects for all solver runs

In [3]:
public class StudySettings {

    public double Oh;

    public int m;
    public double eta0;

    public string caseName;

    public int maxL;

    public double t_end;
    public double dt;
    public int savePeriod;
    public int logPeriod;

    public StudySettings(double OhnesorgeNumber, int mode, double deformParam, int maxLmode, double tend, int savePer = 10, int logPer = 2) {

        Oh = OhnesorgeNumber;
        m = mode;
        eta0 = deformParam;

        caseName = $"m{m}_Oh{Oh.ToString("0.##").Replace(".", "")}_eta{eta0.ToString("0.###").Replace(".", "")}";

        maxL = maxLmode;

        t_end = tend;
        dt = 0.005;

        savePeriod = savePer;   
        logPeriod = logPer;

    }

}

In [4]:
List<StudySettings> allStudies = new List<StudySettings>();

allStudies.Add(new StudySettings(0.1, 2, 0.1, 12, 7.0));    // case 1.1
allStudies.Add(new StudySettings(0.1, 2, 0.2, 12, 7.0));    // case 1.2
allStudies.Add(new StudySettings(0.1, 2, 0.4, 12, 7.0));    // case 1.3
allStudies.Add(new StudySettings(0.1, 3, 0.15, 12, 7.0));    // case 2.1
allStudies.Add(new StudySettings(0.1, 3, 0.4, 24, 7.0));    // case 2.2
allStudies.Add(new StudySettings(0.1, 4, 0.1, 20, 7.0));    // case 3.1
allStudies.Add(new StudySettings(0.1, 4, 0.4, 20, 7.0));    // case 3.2
allStudies.Add(new StudySettings(0.56, 4, 0.05, 20, 4.0));    // case 4

In [5]:
bool[] thirdOrderInit = new bool[] { true, false};

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

foreach(var study in allStudies) {
foreach(bool useThirdOrderInit in thirdOrderInit) {

    if ((study.caseName == "m3_Oh01_eta04" || study.caseName == "m4_Oh01_eta04") && useThirdOrderInit)
        continue; // skipping this case since no reference data is available

    var C = new XNSE_Control();

    int k = 3;
    C.SetDGdegree(k);


    // physical parameters
    // ===================
    C.PhysicalParameters.rho_A = 1;
    C.PhysicalParameters.rho_B = 0.001;
    C.PhysicalParameters.mu_A = study.Oh;
    C.PhysicalParameters.mu_B = study.Oh/1000.0;
    C.PhysicalParameters.Sigma = 1;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;
    

    // set grid
    // ========
    C.SetGrid(Grids[0]);

    
    // initial conditions
    // ==================
    C.InitialValues.Add("Phi", casesPhi[study.caseName]);

    if(useThirdOrderInit) {
        C.AddInitialValue("VelocityX#A", analyticVel[study.caseName].velX);
        C.AddInitialValue("VelocityY#A", analyticVel[study.caseName].velY);
        C.AddInitialValue("VelocityZ#A", analyticVel[study.caseName].velZ);
    }   


    // misc. solver options
    // ====================
    C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    C.NonLinearSolver.ConvergenceCriterion = 1e-9;  
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.MinSolverIterations = 3;


    // level-set
    // =========
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = 1 });


    // Timestepping
    // ============    
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.dtFixed = study.dt;
    C.Endtime = runShortTests ? 0.5 : study.t_end;
    C.NoOfTimesteps = (int)(study.t_end / study.dt);
    

    
    C.PostprocessingModules.Add(new SphericalHarmonicsLogging() { SolverStage = 2, MaxL = study.maxL, RotSymmetric = true, LogPeriod = study.logPeriod });
    C.PostprocessingModules.Add(new DropletMetricsLogging() { SolverStage = 2, AxisSymmetric = true, LogPeriod = study.logPeriod });
    C.PostprocessingModules.Add(new EnergyLogging() { SolverStage = 2, LogPeriod = study.logPeriod });

    C.saveperiod =  study.savePeriod;
    
    //C.TracingNamespaces = "*";

    if(useThirdOrderInit)
        C.SessionName = $"OD3D_{study.caseName}_thirdOrderInit";
    else
        C.SessionName = $"OD3D_{study.caseName}_stagnantInit";

    //Console.WriteLine($"Setting up case: {C.SessionName}");
    
    Controls.Add(C);
    
}
}

Error: (1,6): error CS0246: The type or namespace name 'XNSE_Control' could not be found (are you missing a using directive or an assembly reference?)
(1,40): error CS0246: The type or namespace name 'XNSE_Control' could not be found (are you missing a using directive or an assembly reference?)
(9,17): error CS0246: The type or namespace name 'XNSE_Control' could not be found (are you missing a using directive or an assembly reference?)
(29,15): error CS0103: The name 'Grids' does not exist in the current context
(34,32): error CS0103: The name 'casesPhi' does not exist in the current context
(37,42): error CS0103: The name 'analyticVel' does not exist in the current context
(38,42): error CS0103: The name 'analyticVel' does not exist in the current context
(39,42): error CS0103: The name 'analyticVel' does not exist in the current context
(45,36): error CS0103: The name 'NonLinearSolverCode' does not exist in the current context
(53,34): error CS0103: The name 'BoSSS' does no

In [85]:
foreach (var ctrl in Controls) {
    //Console.WriteLine($"{ctrl.SessionName}");
    Console.WriteLine($"{ctrl.SessionName}: tend = {ctrl.Endtime}");
}

OD3D_m2_Oh01_eta01_thirdOrderInit: tend = 0.1
OD3D_m2_Oh01_eta01_stagnantInit: tend = 0.1
OD3D_m2_Oh01_eta02_thirdOrderInit: tend = 0.1
OD3D_m2_Oh01_eta02_stagnantInit: tend = 0.1
OD3D_m2_Oh01_eta04_thirdOrderInit: tend = 0.1
OD3D_m2_Oh01_eta04_stagnantInit: tend = 0.1
OD3D_m3_Oh01_eta015_thirdOrderInit: tend = 0.1
OD3D_m3_Oh01_eta015_stagnantInit: tend = 0.1
OD3D_m3_Oh01_eta04_stagnantInit: tend = 0.1
OD3D_m4_Oh01_eta01_thirdOrderInit: tend = 0.1
OD3D_m4_Oh01_eta01_stagnantInit: tend = 0.1
OD3D_m4_Oh01_eta04_stagnantInit: tend = 0.1
OD3D_m4_Oh056_eta005_thirdOrderInit: tend = 0.1
OD3D_m4_Oh056_eta005_stagnantInit: tend = 0.1


In [78]:
int NoCtrls = Controls.Count();
NoCtrls

14

## Launch Jobs

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);

    jobs.Add(oneJob);
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-W

## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                    || job.Status == JobStatus.PendingInExecutionQueue
                                    || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [ ]:
int maxDays = 10;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
var FailedSess = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled);
FailedSess

0

In [ ]:
int NoFailed = FailedSess.Count();
NoFailed

In [ ]:
var SuccessSess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful);
SuccessSess

In [ ]:
int NoSuccess = SuccessSess.Count();
NoSuccess

17

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var job in jobs) {
        //var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(job.Name));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine($"No restart session found. {job.Name} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {job.Name}!"); // should not happen
            } 
        }
    }

}

In [ ]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");